# ДЗ1. Воспроизводимость модели: sklearn Pipeline

Курсовой проект ML/DLOps. Анализ данных Inside Airbnb.

Цель работы: объединить предобработку признаков и модель Random Forest в единый sklearn Pipeline, обучить его, оценить качество на тестовой выборке, сохранить в model.pkl и проверить получение предсказаний после загрузки модели.

### Импорт библиотек

In [1]:
!pip install -q boto3

import os
import boto3
import pandas as pd

### Настраиваем доступ к S3

In [2]:
# Получаем ключи из переменных окружения
if not os.getenv("AWS_ACCESS_KEY_ID") or not os.getenv("AWS_SECRET_ACCESS_KEY"):
    try:
        from google.colab import userdata

        os.environ["AWS_ACCESS_KEY_ID"] = userdata.get("AWS_ACCESS_KEY_ID")
        os.environ["AWS_SECRET_ACCESS_KEY"] = userdata.get("AWS_SECRET_ACCESS_KEY")
    except ImportError:
        raise RuntimeError(
            "Необходимо задать AWS_ACCESS_KEY_ID и AWS_SECRET_ACCESS_KEY"
        )

S3_BUCKET = "mlops26-course-bucket"
S3_KEY = "prepared/listings.csv"

s3 = boto3.client(
    "s3",
    endpoint_url="https://storage.yandexcloud.net",
    region_name="ru-central1"
)

print("Клиент S3 создан")


Клиент S3 создан


### Читаем данные и смотрим таблицу

In [3]:
response = s3.get_object(
    Bucket=S3_BUCKET,
    Key=S3_KEY
)

with response["Body"] as body:
    df = pd.read_csv(body)

print("Размер таблицы:", df.shape)

display(df.head())

print("Названия столбцов:")
print(df.columns.tolist())

Размер таблицы: (20151, 13)


,listing_id,snapshot_date,latitude,longitude,accommodates,bathrooms,bedrooms,beds,minimum_nights,neighbourhood_cleansed,property_type,room_type,price
0,2145993,2026-09-17,51.511100,-0.198250,2,1.0,1.0,1.0,4.0,Kensington and Chelsea,Entire rental unit,Entire home/apt,251.75
1,846381013872498937,2026-09-17,51.599540,-0.064500,6,1.5,3.0,4.0,2.0,Haringey,Entire home,Entire home/apt,201.60
2,1230990949123993577,2026-06-19,51.371535,-0.097551,2,1.0,NaN,1.0,1.0,Croydon,Private room in rental unit,Private room,73.00
3,54178450,2026-09-17,51.390740,-0.273480,2,1.0,1.0,1.0,1.0,Kingston upon Thames,Entire condo,Entire home/apt,126.00
4,1297347987655075325,2026-03-21,51.517398,0.012953,4,1.0,1.0,2.0,3.0,Newham,Entire rental unit,Entire home/apt,147.00


Названия столбцов:
['listing_id', 'snapshot_date', 'latitude', 'longitude', 'accommodates', 'bathrooms', 'bedrooms', 'beds', 'minimum_nights', 'neighbourhood_cleansed', 'property_type', 'room_type', 'price']


Буду решать задачу регрессии: прогнозирование стоимости аренды жилья Airbnb.

У нас есть числовая целевая переменная price и признаки, которые потенциально связаны со стоимостью жилья.

## Подготовка данных и разделение на train/test

Для прогнозирования цены используем числовые и категориальные характеристики жилья. Целевая переменная — `price`.

Разделим данные на обучающую и тестовую выборки с помощью `GroupShuffleSplit`, чтобы одно и то же объявление не попадало одновременно в обе выборки.

In [4]:
from sklearn.model_selection import GroupShuffleSplit

# Признаки и целевая переменная
features = [
    "latitude",
    "longitude",
    "accommodates",
    "bathrooms",
    "bedrooms",
    "beds",
    "minimum_nights",
    "neighbourhood_cleansed",
    "property_type",
    "room_type"
]

X = df[features].copy()
y = df["price"].copy()

# Разделение по уникальным объявлениям
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=df["listing_id"])
)

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()

y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

print("Размер train:", X_train.shape)
print("Размер test:", X_test.shape)

train_ids = set(df.iloc[train_idx]["listing_id"])
test_ids = set(df.iloc[test_idx]["listing_id"])

print("Общих объявлений:", len(train_ids & test_ids))


Размер train: (16076, 10)
Размер test: (4075, 10)
Общих объявлений: 0


## Создание sklearn Pipeline

Для предобработки используем ColumnTransformer. Числовые пропуски заполняем медианой, категориальные наиболее частым значением. Категориальные признаки кодируем с помощью OneHotEncoder.

Предобработка и модель Random Forest объединяются в единый Pipeline. Это позволяет выполнять все преобразования и обучение последовательно через один вызов fit().

In [5]:

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor

# Числовые признаки
numeric_features = [
    "latitude",
    "longitude",
    "accommodates",
    "bathrooms",
    "bedrooms",
    "beds",
    "minimum_nights"
]

# Категориальные признаки
categorical_features = [
    "neighbourhood_cleansed",
    "property_type",
    "room_type"
]

# Предобработка числовых признаков
numeric_transformer = SimpleImputer(strategy="median")

# Предобработка категориальных признаков
categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Объединяем преобразования
preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

# Создаём единый Pipeline
pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=100,
        max_depth=15,
        random_state=42,
        n_jobs=-1
    ))
])

pipeline


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  SimpleImputer(strategy='median'),
                                                  ['latitude', 'longitude',
                                                   'accommodates', 'bathrooms',
                                                   'bedrooms', 'beds',
                                                   'minimum_nights']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['neighbourhood_cleansed',
                                                   'property_type',
                                                   'room_type'])])),
                ('model',
                 RandomForestRegressor(max_depth=15, n_jobs=-1,
                                       random_state=42))])

## Обучение и оценка Pipeline

Обучим Pipeline на тренировочной выборке и оценим качество прогноза на тестовых данных. Предобработка признаков и обучение Random Forest выполняются последовательно внутри Pipeline.

In [6]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Обучаем Pipeline
pipeline.fit(X_train, y_train)

# Получаем предсказания на тестовой выборке
y_pred = pipeline.predict(X_test)

# Рассчитываем метрики
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("MAE:", round(mae, 2))
print("RMSE:", round(rmse, 2))
print("R2:", round(r2, 4))

MAE: 102.47
RMSE: 360.57
R2: 0.1592


## Сохранение обученного Pipeline

Сохраним обученный Pipeline в файл `model.pkl` с помощью pickle. В файл попадут и модель Random Forest, и все настроенные преобразования признаков.

In [7]:
import pickle

# Сохраняем обученный Pipeline
with open("model.pkl", "wb") as f:
    pickle.dump(pipeline, f)

print("Pipeline сохранён в model.pkl")


Pipeline сохранён в model.pkl


## Проверка загрузки Pipeline

Загрузим сохранённый Pipeline и получим предсказания на тестовой выборке без повторного обучения. Сравним их с результатами исходной модели.

In [8]:
# Загружаем сохранённый Pipeline
with open("model.pkl", "rb") as f:
    loaded_pipeline = pickle.load(f)

# Получаем предсказания без повторного обучения
y_pred_loaded = loaded_pipeline.predict(X_test)

# Сравниваем предсказания
print("Количество предсказаний:", len(y_pred_loaded))
print("Предсказания совпадают:", np.allclose(y_pred, y_pred_loaded))

# Показываем первые пять результатов
comparison = pd.DataFrame({
    "Фактическая цена": y_test.iloc[:5].values,
    "Прогноз модели": y_pred_loaded[:5].round(2)
})

display(comparison)


Количество предсказаний: 4075
Предсказания совпадают: True


,Фактическая цена,Прогноз модели
0,126.0,159.10
1,339.2,344.08
2,390.0,374.27
3,142.7,251.24
4,352.0,289.35


## Вывод

В работе предобработка признаков и модель Random Forest были объединены в единый sklearn Pipeline. Для числовых и категориальных признаков использовался ColumnTransformer, который позволяет выполнять необходимые преобразования внутри Pipeline.

После обучения модель показала MAE = 102,47, RMSE = 360,57 и R² = 0,1592. Метрики совпали с результатами предыдущего ноутбука, где предобработка и обучение выполнялись отдельно.

Обученный Pipeline был сохранён в `model.pkl` с помощью pickle. После загрузки файла удалось получить все 4 075 предсказаний без повторного обучения. Проверка показала, что предсказания исходной и загруженной модели совпадают.

Таким образом, удалось сохранить не только модель, но и все необходимые преобразования признаков, чтобы использовать Pipeline в дальнейшем.